In [ ]:
!pip install catboost

In [ ]:
import kagglehub
import os
import kagglehub
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import KFold, StratifiedKFold
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, f1_score

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:

data_path = os.path.join(path, 'Q3_data.csv')

df = pd.read_csv(data_path)


In [ ]:
# Task 2: Write your code here:

df.head()

In [ ]:
# Task 3: Write your code here:

df.info()

In [ ]:
# Task 4: Write your code here:

df.describe()

In [ ]:
# Task 1.1: Write your code here:

df.isnull().sum() / df.size * 100

print(f"Percentage of missing values per column:\n{(df.isna().sum() / len(df)) * 100}%")


In [ ]:
# Task 1.2: Write your code here:

missing_percentage = (df.isnull().sum() / len(df)) * 100

missing_data = pd.DataFrame({
    "Columns": missing_percentage.index,
    "Missing_Percentage": missing_percentage.values
    })

missing_data = missing_data[missing_data["Missing_Percentage"] > 0].sort_values(by="Missing_Percentage", ascending=False)

missing_data.head(10)

In [ ]:
# Task 1.3: Write your code here:

high_percent_missing_data = missing_data[missing_data["Missing_Percentage"] > 30]

high_percent_missing_data


In [ ]:
# Task 1.4: Write your code here:

df_clean = df.copy()

for col in high_percent_missing_data.values:

  df_clean = df_clean.drop(col[0], axis=1)

for col in df_clean:

  df_clean[col] = df_clean[col].fillna(0)

print("Missing values remaining:", df_clean.isnull().sum().sum())


In [ ]:
# Task 2: Write your code here:

duplicates = df.duplicated().sum()

print("Number of duplicate rows before dropping:", duplicates)

df.drop_duplicates(inplace=True)

print("Number of duplicate rows after dropping:", duplicates)

In [ ]:
# Task 3: Write your code here:

categorical_cols = list(df_clean.select_dtypes(include=["object"]).columns)

if(categorical_cols == []):

  print(f"We hvae no cateogircal columns:\n")
  print(f"Categorical Columns: {categorical_cols}")

else:
  for col in categorical_cols:
    label_encoder = LabelEncoder()
    df_clean[col] = label_encoder.fit_transform(df_clean[col].astype(str))

df_clean.head()



In [ ]:
# Task 4: Write your code here:

numerical_cols = df_clean.select_dtypes(include=["int64", "float64"]).columns.drop("Target")

scaler = StandardScaler()
df_clean[numerical_cols] = scaler.fit_transform(df_clean[numerical_cols])
df_clean.head()

In [ ]:
# Task 5: Write your code here:

print(df_clean["Target"].value_counts(normalize=True))

df_clean["Target"].hist()
plt.show()

print("Target is imbalanced.")

In [ ]:
# Task 1: Write your code here:

X = df_clean.drop("Target", axis=1)
y = df_clean['Target']

print(f"X Shape {X.shape}")
print(f"y Shape {y.shape}")

In [ ]:
# Task 2,3,4,5: Write your code here:


model_accuracy = []
model_f1_score = []

kfs = StratifiedKFold(n_splits=5, random_state=42, shuffle=True)

model = CatBoostClassifier(verbose=0, n_estimators=320, max_depth=4)

for fold, (train_index, test_index) in enumerate(kfs.split(X, y), start=1):
    print(f"Fold Number: {fold}")

    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    model_accuracy.append(accuracy_score(y_test, y_pred))
    model_f1_score.append(f1_score(y_test, y_pred))


print(f"5-Fold CV Results:")
print(f"MAE:  ${np.mean(model_accuracy):,.2f}")
print(f"MAE:  ${np.mean(model_f1_score):,.2f}")

In [ ]:
# Task 1: Write your code here:

feature_importance = pd.DataFrame({
    'feature': numerical_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)


plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:



In [ ]:
# Task Bonus: Write your code here: